# Preprocessing

In [ ]:
import os
import pandas as pd 
import pyarrow.parquet as pq
import warnings
warnings.filterwarnings("ignore")
from src.aggregation.agg import ElectricityAggregator


## Load Data

In [ ]:
# data paths
DATA_DIR = "mock_data"
OUT_BASE_DIR = "output/data"

MATDATA_PATH = os.path.join(DATA_DIR, "meter_data_500.snappy.parquet")
TARIFF_PATH  = os.path.join(DATA_DIR, "tariff_100.csv")
SURVEY_PATH  = os.path.join(DATA_DIR, "survey_50.csv")

In [ ]:
# Electricity meter data
pf = pq.ParquetFile(MATDATA_PATH)
table = pf.read(columns=["aID", "TIDPUNKT", "TIDPUNKT_DAG", "FORBRUKNING_KWH"])
meter_data = table.to_pandas()

print("Unique households:", len(meter_data["aID"].unique())) # unique households
print("Unique days:", len(meter_data["TIDPUNKT_DAG"].unique())) # unique days
display(meter_data.head(3))

Unique households: 500
Unique days: 782


,aID,TIDPUNKT,TIDPUNKT_DAG,FORBRUKNING_KWH
0,735999166200000851,2024-01-01 00:00:00+00:00,2024-01-01,0.283425
1,735999166200000851,2024-01-01 01:00:00+00:00,2024-01-01,0.300204
2,735999166200000851,2024-01-01 02:00:00+00:00,2024-01-01,0.260809


In [4]:
# Tariff data
tariff_df = pd.read_csv(TARIFF_PATH, sep=",")
print(f"tariff rows: {len(tariff_df):,}") # Tariff number 
display(tariff_df.head(3))

# Survey data
# survey_month_result[month_result["price"] == "high"] = pd.read_csv(SURVEY_PATH)
# print(f"survey rows: {len(survey_month_result[month_result["price"] == "high"]):,}") # Survey number
# display(survey_month_result[month_result["price"] == "high"].head(3))


tariff rows: 100


,Produktnamn,Startdatum,GS1-nr.
0,GENAB Tidsindelad 6 kW Villa,2025-06-01,735999166200288372
1,GENAB Tidsindelad 6 kW Villa,2025-04-01,735999166200186244
2,GENAB Tidsindelad 6 kW Villa,2025-03-01,735999166200055594


# Aggregate electricity meter data 

In [6]:
agg = ElectricityAggregator(meter_data, tariff_df)

Initializing ElectricityAggregator...
Loaded 9,372,500 rows


In [ ]:
# Monthly
month_result = agg.run(
    freq="month",
    agg_method=["top3_mean", "variance", "mean"],
    use_price=True,
    add_user_group_col=[0.25, 0.75],
    output_path=os.path.join(OUT_BASE_DIR, "monthly_agg.parquet")
    )

In [ ]:
month_result.head()

In [ ]:
# Hourly
hour_result = agg.run(
    freq="hour",
    agg_method=["mean"],
    use_price=True,
    add_user_group_col=[0.25, 0.75],
    output_path=os.path.join(OUT_BASE_DIR, "hourly_agg.parquet")
    )

In [7]:
# Week & Weekend
weekday_result = agg.run(
    freq="weekday",
    agg_method=["mean"],
    use_price=True,
    add_user_group_col=[0.25, 0.75],
    output_path=os.path.join(OUT_BASE_DIR, "weekly_agg.parquet")
    )


Electricity aggregation start
Rows: 9,372,500
Frequency: weekday
Aggregation: ['mean']
Include price split (all/high/low): True
Add User groups: [0.25, 0.75]
Merging tariff data...
Tariff merge done (1.91s)
Creating usage groups...
Usage groups created (2.24s)
Aggregating per household...
Aggregation done (8.00s)
Saving parquet...
Saved.
Total runtime: 12.17s
